In [4]:
import requests
import pandas as pd

BASE_URL = "https://clinicaltrials.gov/api/v2/studies"


def fetch_trials(condition, cell_type_keyword, limit=100):
    """Fetch trials from ClinicalTrials.gov API v2."""
    params = {
        "pageSize": min(limit, 100),
        "fields": "NCTId,BriefTitle,OverallStatus,Phase,StudyType,StartDate,Organization,InterventionName,LocationCountry"
    }
    if condition:
        params["query.cond"] = condition
    if cell_type_keyword:
        params["query.term"] = cell_type_keyword

    response = requests.get(BASE_URL, params=params, headers={"Accept": "application/json"}, timeout=45)
    response.raise_for_status()
    return response.json().get("studies", [])


def parse_trials(studies):
    """Parse raw API response into a clean DataFrame."""
    records = []
    for study in studies:
        ps = study.get("protocolSection", {})
        ident = ps.get("identificationModule", {})
        status_mod = ps.get("statusModule", {})
        design = ps.get("designModule", {})
        arms = ps.get("armsInterventionsModule", {})
        contacts = ps.get("contactsLocationsModule", {})

        start_date = status_mod.get("startDate", {})
        start_year = None
        if isinstance(start_date, dict):
            start_year = start_date.get("year")
        elif isinstance(start_date, str):
            try:
                start_year = int(start_date[:4])
            except Exception:
                pass

        phases = design.get("phases", [])
        locations = contacts.get("locations", [])
        countries = list({loc.get("country", "") for loc in locations if loc.get("country")})
        interventions = arms.get("interventions", [])

        records.append({
            "NCT ID": ident.get("nctId", "N/A"),
            "Title": ident.get("briefTitle", "Untitled"),
            "Status": status_mod.get("overallStatus", "UNKNOWN"),
            "Phase": phases[0] if phases else "NA",
            "Study Type": design.get("studyType", "N/A"),
            "Interventions": ", ".join(i.get("name", "") for i in interventions) if interventions else "N/A",
            "Country": countries[0] if countries else "Not Specified",
            "Start Year": start_year,
        })
    return pd.DataFrame(records)
print (parse_trials(fetch_trials("Parkinsons", "stem cell")))

        NCT ID                                              Title  \
0  NCT06944522  A Study to Investigate the Efficacy and Safety...   
1  NCT05573542           Early Signs of Parkinsons Disease in IBS   
2  NCT02795052               Neurologic Stem Cell Treatment Study   
3  NCT03104725  Does N-Acetylcysteine Decrease Spontaneous Oxi...   

                    Status   Phase      Study Type  \
0               RECRUITING  PHASE3  INTERVENTIONAL   
1  ENROLLING_BY_INVITATION      NA   OBSERVATIONAL   
2               RECRUITING      NA  INTERVENTIONAL   
3               TERMINATED  PHASE1  INTERVENTIONAL   

                                    Interventions               Country  \
0                     bemdaneprocel, Sham surgery                Canada   
1                                             N/A                Sweden   
2                 Intravenous and Intranasal BMSC  United Arab Emirates   
3  N-Acetylcysteine, Lumbar Puncture, Fluoroscopy         United States   

  Start